# Cómo está armado este banco

Un Arduino UNO, un sensor magnético de ángulo, un actuador y un motor. Este
notebook recorre el hardware: qué está conectado a qué, por qué está conectado
así, y qué se puede medir con cada parte --incluso mientras alguna de esas partes
todavía no está sobre la mesa.

El banco trabaja en **lazo abierto**: la placa pone un comando de PWM sobre el
actuador y devuelve el ángulo y la corriente, y nada más. No hay controlador a
bordo ni filtros: derivar la velocidad, elegir el signo y ajustar un modelo pasa
de este lado, donde se ve.

Se arma en cuatro pasos, y **cada paso ya sirve para algo**:

| | Configuración | Qué se agrega | Qué habilita |
|---|---|---|---|
| **A** | sólo el sensor | AS5600 en el bus I2C | el ángulo, el desenrollado, el ruido, el ritmo del muestreo |
| **B** | **con** puente en H, sección 2.1 | L298N y su fuente | mover el motor en los dos sentidos |
| **B′** | **con** actuador de un solo cuadrante, sección 2.2 | un transistor y un diodo | mover el motor para un lado, que es lo que tiene este banco |
| **C** | **sin** sensor de corriente | -- | todo lo anterior; `i` queda como un canal que no mide |
| **D** | **con** ACS712 | el sensor de corriente en A0 | ver el consumo&nbsp;⁽¹⁾ |

**B′ es el caso de este banco**, y no es un banco de segunda: la identificación de
la planta sale completa con un solo cuadrante. La sección 2.2 dice qué cambia.

⁽¹⁾ Con reservas: la medición de corriente de este banco **no resuelve este motor**.
La nota al pie de la sección 4 dice exactamente por qué.

Después del recorrido hay dos secciones de código: la **API** con la que se maneja
todo esto desde Python, y un **punto de partida para identificar la planta**, que
es de donde conviene arrancar la práctica.

> ⚠️ **Con el motor conectado, el motor se mueve.** Revisar que el eje esté libre.

> Hay **un solo `dev`**, el de la celda que sigue, y el notebook se recorre de
> arriba abajo. Si se toca el cableado en el medio, volver a correr esa celda.

> **Para orientarse.** Poner `dev` en una celda muestra todo lo que la placa
> tiene: cada parámetro con su valor de ahora, su unidad, si se puede mover y una
> línea de qué es. `dev.describe('ang')` filtra por subsistema, y `dev.<TAB>`
> completa los nombres. No hay ninguna lista escrita de este lado: la placa
> declara la suya al conectarse.

In [ ]:
import sys, os
sys.path.insert(0, '../python')

import numpy as np
import matplotlib.pyplot as plt

import ensayo
from banco_simulado import conseguir_banco

# --- cómo se ven los gráficos de este notebook -------------------------------
AZUL, NARANJA, AQUA, AMARILLO = '#2a78d6', '#eb6834', '#1baf7a', '#eda100'
TINTA, TENUE, NUBE = '#0b0b0b', '#52514e', '#c9c8c3'

plt.rcParams.update({
    'figure.figsize': (9, 3.6), 'figure.dpi': 110,
    'axes.grid': True, 'axes.axisbelow': True, 'grid.color': '#e6e5e1',
    'grid.linewidth': 0.8, 'axes.edgecolor': '#c9c8c3', 'axes.linewidth': 0.8,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.labelcolor': TENUE, 'axes.titlecolor': TINTA, 'axes.titlelocation': 'left',
    'axes.titleweight': 'medium', 'axes.titlepad': 10,
    'xtick.color': TENUE, 'ytick.color': TENUE, 'text.color': TINTA,
    'lines.linewidth': 1.8, 'legend.frameon': False, 'font.size': 10,
})

# El banco de verdad si el cable está enchufado; si no, uno simulado que lo dice.
dev = conseguir_banco(forzar_simulado=os.environ.get('HW_SIMULADO') == '1')
SIMULADO = getattr(dev, 'simulado', False)

# La calibración del sensor de este banco, si está medida. No es tema de este
# notebook --se mide en calibracion.ipynb-- pero sin ella el ángulo llega torcido,
# y una identificación de velocidad hereda la ondulación como si fuera del motor.
if not SIMULADO:
    try:
        import calib
        from bench import CALIBRACION
        if CALIBRACION.exists():
            calib.asegurar(dev, CALIBRACION)
            print('calibracion del sensor aplicada')
        else:
            print('sin calibracion del sensor: el angulo va crudo '
                  '(ver notebooks/calibracion.ipynb)')
    except Exception as exc:
        print(f'no se pudo aplicar la calibracion: {exc}')

## 0. El mapa

En el medio hay un UNO haciendo tres cosas a la vez, y conviene tenerlas separadas
en la cabeza porque cada una tiene su propio reloj:

1. **Muestrea el sensor a 5 kHz**, con el Timer2, período rígido. Una lectura del
   AS5600 por interrupción, sin esperar activamente.
2. **Pone el comando sobre el actuador y arma una fila a 500 Hz**: cada `loop_div`
   muestras --10 por omisión-- lee el ángulo y la corriente.
3. **Emite telemetría a 1 Mbaud**: una fila por período, 25 bytes, el 13 % del
   enlace. Es lo que llega a este notebook como un `DataFrame`.

```
   ┌────────────────────┐
   │    Arduino UNO     │   A4/A5 ──── I2C ─────►  AS5600      (configuración A)
   │                    │
   │  Timer2 → 5 kHz    │   9/6/7 ──── PWM+dir ─►  actuador ─► motor   (B, B′)
   │  filas  → 500 Hz   │
   │  USART  → 1 Mbaud  │   A0    ◄─── analógica   ACS712     (C y D)
   └────────┬───────────┘
            │ USB
            ▼
        Jupyter
```

Los pines, completos:

| Señal | Pin | Hace falta para | Si falta |
|---|---|---|---|
| AS5600 `SDA` | A4 | el ángulo | el ángulo queda congelado, las filas siguen saliendo |
| AS5600 `SCL` | A5 | ídem | ídem |
| AS5600 `VDD` / `GND` | 5V / GND | ídem | ídem |
| `ENA`, o la puerta del transistor | 9 (PWM) | mover el motor | se puede medir todo lo que no requiera movimiento |
| L298N `IN1` | 6 | el sentido, con un puente | con un transistor no va a ningún lado |
| L298N `IN2` | 7 | ídem | ídem |
| Salida del ACS712 | A0 | la corriente | `i` informa el ruido de una entrada al aire, y `bringup()` lo detecta |

**Periféricos de los que el sketch se apropia.** Vale saberlo antes de agregarle
algo al montaje, porque las funciones de Arduino que uno esperaría no están
disponibles:

| Recurso | Para qué | Qué deja de funcionar |
|---|---|---|
| Timer2 | el muestreador de 5 kHz | `analogWrite()` en 3 y 11, `tone()` |
| Timer1 | el PWM del actuador, con su propio TOP | `analogWrite()` en 9 y 10, `Servo` |
| ADC | se maneja a mano, canal 0, contra Vcc | `analogRead()` |
| USART | 1 Mbaud, el protocolo | `Serial` para cualquier otra cosa |
| TWI | `nI2C`, por interrupciones | `Wire` |
| Timer0 | -- | nada: `millis()` y el PWM de 5 y 6 andan como siempre (acá el 6 lo usa `IN1`, como salida digital) |

## 1. Configuración A: sólo el sensor

Sin puente, sin motor y sin medición de corriente. Es la configuración con la que
conviene empezar, y no es un juguete: acá ya se ve el sensor, el desenrollado, el
ruido y el ritmo del muestreo. Lo único que falta es que algo gire solo.

```
   ┌───────────────┐                            ┌─────────────┐
   │  Arduino UNO  │  A4 ──── SDA ──────────────│   AS5600    │
   │               │  A5 ──── SCL ──────────────│  (plaqueta) │
   │  Timer2:5 kHz │  5V ──── VDD ──────────────│             │
   │  USB: 1 Mbaud │ GND ──── GND ──────────────│             │
   └───────────────┘                            └──────┬──────┘
                                                       │
                                                imán diametral
                                             1-2 mm sobre el chip
```

**El imán es parte del sensor, no un accesorio.** Tiene que estar magnetizado
**diametralmente** --los polos enfrentados a lo ancho, no norte arriba y sur
abajo--, girar sobre la cara del chip a un par de milímetros, y estar centrado con
el eje de giro. La hoja de datos pide un cuarto de milímetro de excentricidad; lo
que sobra de eso no aparece como ruido sino como una función fija del ángulo que se
repite vuelta tras vuelta. Eso es el tema completo de `calibracion.ipynb`.

**Las plaquetas comerciales de AS5600 traen su propio regulador y los pull-ups del
bus**, así que se enchufan a 5 V y andan. Un chip pelado en modo 3,3 V necesita
adaptación de niveles y sus propias resistencias de pull-up.

**El AS5600 tiene un filtro adentro**, y viene mal puesto para esto: arranca en
16x, que son 2,2 ms de retardo. El sketch lo pasa a 2x, 0,286 ms, una vez al
arrancar. Ese retardo se identificaría después como si fuera un tiempo muerto del
motor, así que no es una perilla: se lo deja al mínimo y listo.

**El muestreo no se detiene si el sensor no está.** Si el AS5600 no contesta, el
muestreador pasa a sondear el bus dos veces por segundo en lugar de cinco mil, y
todo lo demás --el período, la telemetría, los parámetros-- sigue igual. Sirve para
probar la cadena completa (compilar, grabar, capturar, graficar) antes de tener el
sensor sobre la mesa.

`bringup()` es la verificación del equipo, subsistema por subsistema. Con
`motor=False` saltea todo lo que haría girar el eje, que es exactamente lo que
corresponde en esta configuración.

In [ ]:
dev.bringup(motor=False)

Lo que hay que mirar en esa salida, en orden:

- **`muestreo`**: la frecuencia real contra la nominal, medida con el reloj de esta
  computadora y no con el contador de la placa --que avanza una vez por período
  *atendido* y por eso daría siempre por bueno lo que hay que detectar.
- **`margen de tiempo`**: cuánto del período se consume en el peor caso. Sin margen
  alguno la placa está por empezar a perder filas.
- **`iman`**: el AGC del propio AS5600. Contra 0 o contra 255 el imán está a la
  distancia equivocada, y ahí no hay calibración que arregle nada.
- **`bus i2c`**: errores de transferencia y desbordes. Un puñado de desbordes por
  segundo es normal --el diagnóstico del imán lee un registro extra dos veces por
  segundo y esa lectura no entra en 200 us--; lo que no es normal es que el bus no
  llegue de manera sostenida.

Y ahora el sensor en vivo. Hay que **girar el imán con la mano** mientras la celda
corre. `y_uw` es el ángulo *desenrollado*: sigue contando a través de la vuelta de
4096 cuentas en lugar de saltar a cero, así que un eje que gira sin parar da una
recta que no para. `y_raw` es la cuenta cruda, adentro de la vuelta: la que salta.

In [ ]:
df = dev.capture(3.0)

fig, (a, b) = plt.subplots(2, 1, sharex=True, figsize=(9, 5))
a.plot(df['t'], df['y_uw'] - df['y_uw'].iloc[0], color=AZUL)
a.set_ylabel('y_uw [grados]')
b.plot(df['t'], df['y_raw'], lw=0.8, color=TENUE)
b.set_ylabel('y_raw [grados]'); b.set_xlabel('t [s]')
a.set_title('Girar el imán con la mano')
plt.show()

print(f'se movió {df["y_uw"].max() - df["y_uw"].min():.1f} grados')
print(f'ruido entre muestras consecutivas: {df["y_uw"].diff().std():.3f} grados '
      f'({df["y_uw"].diff().std() / 0.0879:.2f} cuentas)')

## 2. Configuración B: con actuador

Ahora el motor. El UNO no maneja un motor: maneja una llave, y la llave maneja al
motor con su propia fuente. Primero el caso completo --un puente en H, que acciona
en los dos sentidos-- y después, en 2.2, el de un solo cuadrante, que es el de este
banco.

### 2.1 Un puente en H: los dos sentidos

```
   ┌───────────────┐                     ┌──────────────────┐
   │  Arduino UNO  │  9 ──── ENA ────────│      L298N       │      ┌─────────┐
   │               │  6 ──── IN1 ────────│   puente en H    │ OUT1─│  motor  │
   │  Timer1: PWM  │  7 ──── IN2 ────────│                  │ OUT2─│ + imán  │
   │               │ GND ─────┬──────────│ GND       +Vmot  │      └─────────┘
   └───────────────┘          │          └───────┬─────┬────┘
                              └── masa común ────┘     │
                                                       └── fuente del motor
                                                           (NO el USB)
```

**Tres reglas de cableado, y las tres se pagan caras si se saltean.** La fuente del
motor es propia y no el USB: el UNO da la lógica, nunca la potencia. Las masas van
unidas, porque si no las señales de control no tienen contra qué medirse. Y el
actuador es el único que toca los bornes del motor.

**El reparto de los tres pines no es arbitrario.** `ENA` lleva la *magnitud* por
PWM y el par `IN1`/`IN2` el *sentido*. Lo que eso compra es que el sketch pueda
**apagar el puente antes de cambiar de sentido**: baja `ENA` --que abre las cuatro
llaves con una sola escritura--, recién entonces mueve `IN1` e `IN2`. Con el
comando en cero `ENA` queda en bajo, el puente abierto y el motor en punto muerto:
**no frena el eje, sólo deja de empujarlo**.

**El PWM va a 1 kHz, y es una concesión al L298N.** En abstracto conviene modular
más rápido, pero es un puente de Darlington bipolares: cae unos 2 V y tarda unos
2 us en conmutar, y a 20 kHz lo que se pierde en cada transición se lleva una
fracción grande de un tiempo de encendido que ya venía escaso. **Medido: a 20 kHz
el motor no arranca y a 1 kHz anda.** Es una constante del sketch (`PWM_TOP`).

**El signo no es un parámetro.** Si un comando positivo hace *bajar* el ángulo, es
de qué lado están los cables del motor y de qué lado mira el imán; en lazo abierto
eso se resuelve al procesar, con `ensayo.signo()`, y no en la placa.

**Si este banco vuelve a tener un L298N**, hay una manera de accionarlo que deja la
planta lineal con un solo comando: frenar en lugar de soltar, con `ENA` en alto y
el PWM sobre la entrada del sentido --la otra en cero--, de modo que en la parte
baja del ciclo el puente cortocircuita el motor en vez de abrirlo. Lo que no
conviene es modular bipolar a 1 kHz.

### 2.2 Un solo cuadrante: el actuador de este banco

Un transistor y un diodo, lo mínimo que mueve un motor:

```
                       +Vmot
                         │
                  ┌──────┴──────┐
               [motor]     [diodo de rueda libre]   el cátodo (la banda) va
                  │             │                   del lado de +Vmot
                  └──────┬──────┘
                         │
   UNO 9 ──[1k]──► G ┌───┴────┐
                     │ llave  │  del lado de masa
                     └───┬────┘
                         │
                        GND ──── común con el UNO
```

**El diodo no es opcional**: al cortar la corriente, la inductancia del motor la
sigue empujando, y sin un camino de retorno esa energía aparece como un pico de
tensión sobre el transistor.

**Qué cambia respecto del puente**, y es lo que cualquier modelo de este banco tiene
que contemplar:

- **Empuja y no frena.** Cuando el transistor se abre la corriente se descarga por
  el diodo, y cuando se extingue no hay nada que la invierta. Bajar la velocidad lo
  hace el rozamiento solo, así que **bajar tarda bastante más que subir**.
- **El sentido está fijado en cobre.** `IN1` e `IN2` no van a ningún lado: un
  comando negativo sale por el pin 9 con el mismo módulo y **empuja para el mismo
  lado**. La celda de abajo lo muestra.
- **Con el comando en cero el eje sigue girando** muchos segundos. Todo ensayo tiene
  que esperar a que pare: `ensayo.esperar_quieto(dev)`.
- **La corriente se corta antes de terminar el período** a velocidades medias --la
  inductancia del motor es chica contra un período de 1 ms--, y eso hace que la
  velocidad no sea proporcional al comando. La sección 6 lo mide.

**Qué sigue funcionando completo**, que es casi todo: **la sección 6 entera**, y
**la calibración del sensor** (`calibracion.ipynb`), que mide soltando el motor y
dejándolo desacelerar por rozamiento, que es justo lo que sabe hacer un cuadrante.

In [ ]:
print('lo que hace el eje con un comando positivo y con uno negativo:')
for u in (+150, -150):
    ensayo.esperar_quieto(dev)
    dev.ctl_uff = u
    dev.capture(2.0, warn=False)                       # que llegue al régimen
    df = dev.capture(0.5, warn=False)
    dev.rest()
    _, w = ensayo.velocidad(df, ventana=0)
    print(f'  ctl_uff = {u:+4d}   →   u = {df["u"].mean():+6.1f} en la placa, '
          f'{np.mean(w):+7.1f} rad/s')

print('\nsi las dos velocidades tienen el mismo signo, el actuador es de un solo')
print('cuadrante: el sentido está en los cables y no en el comando')

### 2.3 La verificación completa

`bringup()`, ahora con el motor. Espera a que el eje esté quieto, lo acciona unas
décimas de segundo, y se fija que haya vuelto movimiento o corriente. Si un comando
positivo hace bajar el ángulo lo dice como nota --no es una falla: se resuelve al
procesar--, y si el pico de corriente del arranque no se despega del ruido del
canal, también lo dice.

In [ ]:
dev.bringup()                    # OJO: esto mueve el motor

### El escalón en lazo abierto

`dev.step()` mantiene `pre` segundos, cambia el parámetro y mantiene `post`
segundos más. La placa informa el tick exacto en el que cayó el cambio, así que
**`t = 0` es el escalón mismo con precisión de una muestra**, y la fluctuación de
temporización de esta computadora nunca entra en los datos.

Antes, `ensayo.esperar_quieto()`: con el eje todavía girando del ensayo anterior,
lo que se mide es la cola de ése.

In [ ]:
ensayo.esperar_quieto(dev)
dev.zero_current()               # el cero del sensor de corriente, actuador abierto

df = dev.step('ctl_uff', 200, pre=0.3, post=2.0, back=0)
datos = ensayo.normalizar(df)    # t [s], u [%], theta [rad], omega [rad/s], i [A]

fig, (a, b, c) = plt.subplots(3, 1, sharex=True, figsize=(9, 6.4))
a.plot(datos['t'], datos['omega'], color=AZUL);          a.set_ylabel('ω [rad/s]')
b.plot(datos['t'], datos['u'], color=TENUE);             b.set_ylabel('u [%]')
c.plot(datos['t'], datos['i'] * 1000, color=NARANJA, lw=1); c.set_ylabel('i [mA]')
c.set_xlabel('t [s]')
for ax in (a, b, c):
    ax.axvline(0, color=TENUE, lw=0.9, ls='--')
a.set_title('Escalón en lazo abierto: u = 0 → 78 %')
plt.show()

print(f'velocidad al final: {datos["omega"].iloc[-50:].mean():.1f} rad/s, '
      f'signo del banco {ensayo.signo(df):+d}')

## 3. Configuración C: sin medición de corriente

Es la configuración de la mayoría de los bancos, y no se pierde casi nada. Sin nada
conectado a A0:

- el canal `i` sigue apareciendo en cada fila, informando el ruido de una entrada
  al aire;
- `bringup()` **lo detecta y lo dice**: la entrada queda contra un riel del ADC, y
  eso es una *ausencia*, no un offset. La diferencia importa, porque calibrar un
  cero ahí dejaría un canal que informa ceros perfectos sin haber medido nada;
- lo que no se puede hacer es mirar el pico de arranque, ni usar la corriente como
  evidencia de que el motor está haciendo algo.

Todo el resto --el ángulo, la identificación de la planta mecánica-- funciona igual.

## 4. Configuración D: con un ACS712

El ACS712 es un sensor de corriente de efecto Hall: la corriente atraviesa una
pista interna, el campo que genera se mide del otro lado de un aislamiento, y la
salida es una tensión analógica que va a A0.

**Dónde se lo inserta cambia lo que mide.** En la alimentación del actuador mide el
consumo, que es siempre positivo; en serie con un borne del motor mide la corriente
del motor, que con un solo cuadrante también es siempre positiva.

**El ADC lee contra Vcc**, y no por gusto: un ACS712 es bipolar y ratiométrico,
reposa en la mitad de su alimentación, y medido contra la misma tensión que lo
alimenta queda en media escala por construcción. Contra la referencia interna de
1,1 V saturaría en reposo.

| | 1 cuenta de 12 bits | Con 185 mV/A |
|---|---|---|
| clon con LGT8F328P (ADC de 12 bits) | 1,22 mV | 6,6 mA |
| UNO (ADC de 10 bits, corrido dos lugares) | 4,89 mV | 26,4 mA |

**Hay un cero que se puede medir y una ganancia que no.** El cero tiene una
condición conocida --con el actuador abierto no circula corriente-- así que
`dev.zero_current()` lo mide y lo resta, y `bringup()` ya lo corre solo. La
ganancia es otra cosa: haría falta una **corriente conocida**. Un tester en serie
con el motor, una vez; el número vive en `SENSE_MV_PER_A`, en el sketch.

La celda que sigue mira la corriente durante un escalón y la compara con su propio
ruido en reposo.

In [ ]:
ensayo.esperar_quieto(dev)
dev.zero_current()

reposo = dev.capture(1.0, warn=False)
ruido = reposo['i'].std()

df = dev.step('ctl_uff', 200, pre=0.3, post=1.5, back=0)
dev.rest()

# Un promedio de 100 ms es lo mejor que se puede pedirle a este canal: una
# muestra suelta es casi toda ruido.
n = int(0.1 / dev.dt)
suave = np.convolve(df['i'], np.ones(n) / n, mode='same')

plt.plot(df['t'], df['i'], color=NUBE, lw=0.8, label='cada muestra')
plt.plot(df['t'], suave, color=NARANJA, label='promedio de 100 ms')
plt.axvline(0, color=TENUE, lw=0.9, ls='--')
plt.axhline(0, color=TENUE, lw=0.9)
plt.xlabel('t [s]'); plt.ylabel('i [mA]'); plt.legend()
plt.title('Corriente durante el escalón')
plt.show()

regimen = df[df['t'] > 1.0]['i'].mean()
print(f'ruido en reposo    {ruido:.0f} mA RMS por muestra')
print(f'régimen            {regimen:+.0f} mA de promedio')
print(f'pico               {df["i"].max():.0f} mA, que contra el ruido son '
      f'{df["i"].max() / ruido:.1f} desvíos')
if df['i'].max() < 5 * ruido:
    print('\nel pico no se despega de cinco veces el ruido: este canal NO resuelve la '
          'corriente de este motor. Sirve para ver que hay algo, no para medirlo.')

### ⁽¹⁾ Nota al pie: por qué esta medición no resuelve este motor

Es la parte del banco que hay que mirar con desconfianza, y vale la pena que quede
escrito por qué. Ninguna es un cable flojo:

1. **El motor consume decenas de mA y el sensor es de amperes.** Un ACS712 de 5 A
   da 185 mV/A: 50 mA son 9 mV, que en el UNO no llegan a dos cuentas del ADC.
2. **El ruido del sensor es más grande que la señal.** Medido en este banco: 91 mA
   RMS por muestra, del orden de toda la corriente de régimen. Promediando se baja,
   pero promediar es filtrar, y un filtro largo borra justo el transitorio que se
   quería ver.
3. **Se muestrea la forma de onda en una fase fija.** El ADC convierte una vez por
   tick de 200 us y el PWM va a 1 kHz, del mismo cristal: la fase **no deriva
   nunca**. Lo que se mide no es el promedio de la corriente sino siempre el mismo
   instante del período, y con un solo cuadrante la corriente se extingue antes de
   que termine: el sesgo depende del ciclo de trabajo y no se promedia con el tiempo.
4. **La ganancia no es verificable desde acá.** Con el actuador abierto la corriente
   es cero y eso alcanza para el offset, pero no hay ninguna condición conocida de
   corriente *distinta* de cero. Hasta que alguien ponga un tester en serie, la
   escala de este canal es una hipótesis.

**Qué se puede hacer igual con este canal**: ver *que* hay corriente, y comparar
órdenes de magnitud. **Qué no**: usarlo para nada cuantitativo --un $K_t$, un par de
rozamiento, un modelo eléctrico--. Y así como está **es un buen ejercicio**: medir
con un tester, comparar, y decidir qué sensor haría falta.

## 5. La API

Todo lo de arriba se maneja con tres ideas: los parámetros son atributos, las
capturas devuelven un `DataFrame`, y las unidades son unidades reales.

### Conseguir el banco

```python
from bench import *

dev = sync_board()        # compila si cambió el sketch, graba si cambió el binario,
                          # y reabre el enlace (lo que resetea la placa)
dev = sync_board_cal()    # lo mismo, más la calibración del sensor de este banco
```

Este notebook usa `conseguir_banco()`, que hace lo mismo y **cae a un banco
simulado si no hay placa**, avisando fuerte que lo que se ve es un modelo. Es lo
que permite dar la clase con el cable desenchufado.

### Los parámetros son atributos, y la placa los enumera sola

Se leen y se escriben como cualquier atributo, y cada uno viaja a la placa en el
momento: `dev.ctl_uff = 120`, `print(dev.ang_agc)`. Poner `dev` en una celda
muestra la tabla que realmente hay:

```python
dev                      # la tabla entera, agrupada por subsistema
dev.describe('ang')      # sólo el sensor de ángulo
```

| Prefijo | De qué es | | Clase | Qué significa |
|---|---|---|---|---|
| `ctl_` | el comando | | perilla | se fija; es una decisión del experimento |
| `ang_` | el sensor de ángulo | | lectura | la placa la publica; escribirla no significa nada |
| `cur_` | la medición de corriente | | cuenta | un total; ponerlo en cero empieza a contar de nuevo |
| `loop_` | el reloj del muestreo | | | |

### Lo que hace el banco, y lo que hace `ensayo`

```python
dev.rest()                    # comando en cero: donde termina todo experimento
dev.zero_current()            # la corriente de ahora es el cero (actuador abierto)
dev.bringup()                 # la verificación completa del equipo

ensayo.esperar_quieto(dev)    # comando en cero, y esperar a que el eje pare
ensayo.velocidad(df)          # (t, ω) en rad/s: diferencia y promedio móvil centrado
ensayo.signo(df)              # +1 si un comando positivo sube el ángulo, -1 si no
ensayo.normalizar(df)         # t [s], u [%], theta [rad], omega [rad/s], i [A]
ensayo.guardar(df, ruta)      # lo mismo, a un CSV
ensayo.cargar(ruta)
```

In [ ]:
dev

### Capturar

`dev.capture(segundos)` emite y devuelve un `DataFrame` con una fila por período.
Las columnas son los canales de la placa, ya en unidades reales:

| Columna | Qué es |
|---|---|
| `t` | segundos desde el arranque de la captura, o desde el escalón si hubo uno |
| `y_raw` | el ángulo crudo del sensor, adentro de la vuelta y sin corregir |
| `y_uw` | el ángulo desenrollado, en grados, corregido si `ang_cal = 1` |
| `u` | el comando que salió al actuador, de -255 a 255 |
| `i` | la corriente, en mA |

Y en `df.attrs` viene **la salud de esa captura en particular**: los contadores se
ponen en cero antes de arrancar y se leen al terminar. Toda captura además avisa si
perdió períodos o descartó filas, porque una serie temporal a la que le faltan
muestras se ve exactamente igual que una sana hasta que uno va a fijarse.

```python
df = dev.capture(2.0)                                       # dos segundos y nada más
df = dev.capture(2.0, events=[(0.5, 'ctl_uff', 200)])       # con un cambio a los 0,5 s
df = dev.step('ctl_uff', 200, pre=0.3, post=0.7, back=0)    # un escalón, con t = 0 en el escalón
```

In [ ]:
df = dev.capture(1.0, warn=False)

print('columnas:', ', '.join(df.columns))
print(f'{len(df)} filas en {df.attrs["wall"]:.2f} s de reloj de pared')
print()
print('salud de esta captura:')
print(f'  períodos perdidos      {df.attrs["missed"]}')
print(f'  filas descartadas      {df.attrs["drops"]}')
print(f'  peor retardo           {df.attrs["maxlate"]} us de {df.attrs["dt_us"]:.0f} us '
      f'({df.attrs["maxlate"] / df.attrs["dt_us"]:.0%} del período)')
print(f'  desbordes del sensor   {df.attrs["sovr"]}')
print(f'  errores del bus i2c    {df.attrs["serr"]}')
print(f'  estado del imán        AGC {df.attrs.get("agc")}, campo {df.attrs.get("mag")}')

## 6. Punto de partida para identificar la planta

Hasta acá el banco está descrito. Lo que sigue es el arranque de una práctica de
identificación: medir la planta, escribir un modelo, y verificarlo prediciendo una
medición que todavía no se hizo. Todo usa comandos positivos, así que corre igual
con un solo cuadrante.

**Qué planta.** Lo que el comando mueve es la velocidad, no la posición. Un motor de
continua con carga inercial, visto desde el PWM y alrededor de un punto de
operación, se parece a un primer orden:

$$\frac{\Omega(s)}{U(s)} = \frac{K}{1 + s\,\tau}$$

con `K` en (rad/s)/% y `tau` dominada por la inercia y el rozamiento. La posición es
la integral de eso.

**Y lo que no es lineal**, que en este banco no es un detalle: la zona muerta de
abajo, la saturación de arriba, y --por el actuador de un solo cuadrante-- una curva
estática que se dobla. Por eso conviene medir en este orden: primero la curva
estática, que dice dónde vale el modelo, y después el transitorio.

### 6.1 La curva estática: la ganancia y la zona muerta

Un comando fijo, esperar el régimen, anotar la velocidad.

In [ ]:
ensayo.esperar_quieto(dev)

comandos = np.arange(0, 256, 25)
medidas = []

for u in comandos:
    dev.ctl_uff = int(u)
    dev.capture(3.0, warn=False)                      # que llegue al régimen, y se tira
    df = dev.capture(0.5, warn=False)                 # ésta es la que cuenta
    # El promedio de las diferencias sobre toda la ventana es exactamente
    # (último ángulo - primero) / tiempo: la velocidad media, sin miedo al ruido.
    _, w = ensayo.velocidad(df, ventana=0)
    medidas.append(np.mean(w))

dev.rest()

# El signo es del cableado, no de la planta: se lo saca del punto más rápido.
SIGNO = 1.0 if medidas[int(np.argmax(np.abs(medidas)))] >= 0 else -1.0
vel = SIGNO * np.array(medidas)
pct = comandos * 100 / 255

mueve = vel > 0.05 * vel.max()
u_muerto = pct[mueve][0]

plt.plot(pct, vel, 'o-', color=AZUL)
plt.axvline(u_muerto, color=TENUE, lw=0.9, ls='--')
plt.annotate(f'arranca antes de {u_muerto:.0f} %', (u_muerto, vel.max() * 0.3),
             xytext=(8, 0), textcoords='offset points', color=TENUE)
plt.xlabel('u [%]'); plt.ylabel('ω en régimen [rad/s]')
plt.title('Curva estática')
plt.show()

K_local = np.diff(vel) / np.diff(pct)
print('ganancia local entre puntos, (rad/s)/%:')
for a, b, k in zip(pct[:-1], pct[1:], K_local):
    print(f'  {a:5.1f} → {b:5.1f} %   {k:6.2f}')

### 6.2 El escalón: la constante de tiempo

Ahora el transitorio, **empezando por encima de la zona muerta**: un escalón que
arranca en cero mezcla el arranque con la dinámica que se quiere medir. Y
**empezando desde el régimen**: se espera en `U0` antes del escalón.

`tau` se estima por el cruce del 63,2 % del salto, que es lo que se hace a mano
sobre una pantalla, y después se dibuja el primer orden encima para ver cuánto se
parece.

In [ ]:
U0, U1 = 102, 204                # 40 % → 80 %

ensayo.esperar_quieto(dev)
dev.ctl_uff = U0
dev.capture(4.0, warn=False)                           # que llegue al régimen de U0
esc = dev.step('ctl_uff', U1, pre=0.5, post=4.0, back=0)
dev.rest()

d = ensayo.normalizar(esc)
t, w = d['t'].to_numpy(), d['omega'].to_numpy()

w0 = np.mean(w[(t > -0.45) & (t < -0.02)])             # régimen antes del escalón
w1 = np.mean(w[t > t.max() - 0.5])                     # régimen después
cruce = w0 + 0.632 * (w1 - w0)

t_post, w_post = t[t > 0], w[t > 0]
k = int(np.argmax(w_post >= cruce))
tau = np.interp(cruce, w_post[k-1:k+1], t_post[k-1:k+1]) if k > 0 else t_post[0]

modelo = w0 + (w1 - w0) * (1 - np.exp(-np.clip(t, 0, None) / tau))

plt.plot(t, w, lw=1, color=NUBE, label='medido')
plt.plot(t, modelo, color=NARANJA, label=f'primer orden, tau = {tau*1e3:.0f} ms')
plt.axhline(cruce, color=TENUE, lw=0.9, ls=':')
plt.axvline(tau, color=TENUE, lw=0.9, ls='--')
plt.xlabel('t [s]'); plt.ylabel('ω [rad/s]'); plt.legend()
plt.title('Escalón de u = 40 % → 80 %')
plt.show()

K_esc = (w1 - w0) / 40.0
print(f'velocidad   {w0:.1f} → {w1:.1f} rad/s')
print(f'ganancia    K = {K_esc:.2f} (rad/s)/%')
print(f'constante   tau = {tau*1e3:.0f} ms')
print(f'\nmodelo:  Omega(s)/U(s) = {K_esc:.2f} / (1 + {tau:.3f} s)')

### 6.3 Verificar el modelo contra una medición que no se usó para ajustarlo

Un modelo ajustado sobre unos datos siempre se parece a esos datos. La pregunta es
si le acierta a **otro** escalón. Acá, a propósito, uno para abajo: con un solo
cuadrante bajar no es subir al revés.

In [ ]:
ensayo.esperar_quieto(dev)
dev.ctl_uff = U1
dev.capture(4.0, warn=False)
val = dev.step('ctl_uff', U0, pre=0.5, post=6.0, back=0)
dev.rest()

dv = ensayo.normalizar(val)
tv, wv = dv['t'].to_numpy(), dv['omega'].to_numpy()

# La predicción sale SÓLO del modelo del escalón de subida.
pred = w1 + (w0 - w1) * (1 - np.exp(-np.clip(tv, 0, None) / tau))

plt.plot(tv, wv, lw=1, color=NUBE, label='medido')
plt.plot(tv, pred, color=AQUA, label='predicho por el modelo de la subida')
plt.axvline(0, color=TENUE, lw=0.9, ls='--')
plt.xlabel('t [s]'); plt.ylabel('ω [rad/s]'); plt.legend()
plt.title('Predicción contra medición: u = 80 % → 40 %')
plt.show()

err = wv[tv > 0] - pred[tv > 0]
print(f'error RMS {np.sqrt(np.mean(err**2)):.1f} rad/s sobre un salto de {w1 - w0:.1f}')

### 6.4 De acá en adelante

Lo de arriba es el piso: un primer orden y su verificación. Las preguntas que siguen
son las que vuelven interesante la práctica:

- **¿`tau` y `K` dependen del punto de operación?** Repetir 6.2 con escalones chicos
  alrededor de varios comandos. Con un actuador de un solo cuadrante la respuesta es
  que sí, y mucho.
- **¿Y de la dirección?** 6.3 ya lo insinúa: bajar tarda más que subir, porque para
  bajar el actuador deja de empujar y frena el rozamiento solo.
- **¿Hace falta un tiempo muerto?** Ajustar un FOPDT por mínimos cuadrados y ver
  cuánto vale; y preguntarse de dónde saldría, en un banco donde el sensor tiene
  0,3 ms de retardo.
- **¿Qué modelo no lineal explica todo junto?** La curva estática, la asimetría y la
  dependencia con el punto de operación salen de la misma física: la corriente que
  no se invierte y se extingue antes de terminar el período.

Y dos advertencias que ya aparecieron y conviene repetir acá, porque son las que
arruinan una identificación sin dejar rastro:

**El sensor sin calibrar mete una ondulación de velocidad que parece del motor.**
Enganchada al ángulo y repetida vuelta tras vuelta, al derivar sale como una
oscilación perfectamente creíble. `calibracion.ipynb` la mide y la corrige.

**Un ensayo que no espera al eje mide al anterior.** Con el comando en cero el motor
sigue girando muchos segundos. `ensayo.esperar_quieto(dev)` antes de cada uno.